# Hybrid Evaluation: Tabular vs Tabular + Image Embeddings

## Objective

Test whether image-derived information improves donor cornea grading beyond the existing clinical and endothelial morphology features, in the primary scenario (AI-assisted expert grading, all 24 features available).

## Protocol

All settings below were fixed before any results were inspected.

| Element | Setting |
|---|---|
| Cohort | Labelled tissues with an image (1,009) |
| Evaluation | 5 repeats × 5 donor-grouped stratified folds from `donor_grouped_folds.csv` (notebook 06) |
| Classifier | Tuned Random Forest from `05_model_selection_and_evaluation.ipynb`, identical for every arm |
| A. Tabular | 24 features → `ms.make_preprocessor()` → RF |
| B. Hybrid | Tabular preprocessing + image embeddings → StandardScaler → PCA (16 components) → RF |
| Image extractor | ImageNet ResNet-50, frozen, full-resolution 1296×972 input, global average pooling (2048-d) |
| Reference arm | Image embeddings only (PCA 16) → RF, to check that the embeddings carry signal |
| Sensitivity | Hybrid with 8 and 32 PCA components; reported, not used for selection |
| Primary metric | Macro F1; balanced accuracy and accuracy also reported |
| Comparison | Paired differences on identical held-out donors; repeat-level summaries and a corrected repeated-CV t-test |

### Leakage safeguards

- Donor groups never cross fold boundaries (asserted below).
- The image extractor is pretrained on ImageNet and frozen. It is never fitted to this dataset or its labels, so extracting embeddings once for all images is leakage-free.
- Imputation, scaling, one-hot encoding and PCA are fitted inside each training fold through a pipeline.
- No hyperparameters are tuned here. The old holdout and shared-split test results play no role in model selection.

In [1]:
import random
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from scipy import stats
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=UserWarning)

sys.path.insert(0, str(Path("../../scripts").resolve()))
import modeling_setup as ms

SEED = ms.RANDOM_STATE
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IMAGE_DATA_DIR = ms.PROCESSED_DIR.parent.parent / "image" / "data" / "processed"
EMBEDDING_PATH = IMAGE_DATA_DIR / "embeddings" / "resnet50_imagenet_fullres.npz"
OOF_PATH = ms.PROCESSED_DIR / "hybrid_oof_predictions.csv"

PRIMARY_PCA = 16
SENSITIVITY_PCA = [8, 32]

# Tuned in 05_model_selection_and_evaluation.ipynb; identical for every arm.
RF_PARAMS = dict(
    n_estimators=300,
    max_depth=15,
    min_samples_split=5,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=SEED,
    n_jobs=-1,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


## 1. Cohort and Donor-Grouped Folds

In [2]:
def normalize_id(value):
    return " ".join(str(value).split()).replace("NEB- ", "NEB-").casefold()

labelled = ms.load_labelled_data()
folds = ms.get_donor_folds(labelled)  # reads the file saved by notebook 06

manifest = pd.read_csv(IMAGE_DATA_DIR / "clean_image_manifest.csv")
review_ids = {normalize_id(i) for i in ["NEB-2024-08-16259 R", "NEB-2025-09-17471 R"]}
manifest["Match_ID"] = manifest["Tissue_ID"].map(normalize_id)
manifest = manifest[~manifest["Match_ID"].isin(review_ids)]

labelled["Match_ID"] = labelled["Tissue_ID"].map(normalize_id)
cohort = (
    labelled
    .merge(manifest[["Match_ID", "Selected_Image"]], on="Match_ID",
           how="inner", validate="one_to_one")
    .merge(folds, on="Tissue_ID", how="left", validate="one_to_one")
    .reset_index(drop=True)
)

assert cohort["Donor_Group"].notna().all()
assert cohort["Selected_Image"].map(lambda p: Path(p).is_file()).all()

y = cohort[ms.TARGET].to_numpy()
print("Cohort tissues:", len(cohort))
print("Donors:", cohort["Donor_Group"].nunique())
print("Grades:", cohort[ms.TARGET].value_counts().to_dict())

Cohort tissues: 1009
Donors: 770
Grades: {'A+': 530, 'A': 314, 'B': 165}


## 2. Image Embeddings

ResNet-50 runs in inference mode on each full-resolution grayscale image (replicated to three channels, ImageNet normalization). Global average pooling makes the network accept the native 1296×972 input, so cell-level texture is not lost to downscaling. Embeddings are cached so the extraction runs once.

In [3]:
from torchvision.models import resnet50, ResNet50_Weights

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


def load_image(path):
    with Image.open(path) as img:
        array = np.asarray(img.convert("L"), dtype=np.float32) / 255.0
    return torch.from_numpy(array)[None].repeat(3, 1, 1)


@torch.inference_mode()
def extract_embeddings(paths, batch_size=4):
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = torch.nn.Identity()
    model.eval().to(device)

    mean, std = IMAGENET_MEAN.to(device), IMAGENET_STD.to(device)
    output = []
    for start in range(0, len(paths), batch_size):
        batch = torch.stack([load_image(p) for p in paths[start:start + batch_size]])
        batch = (batch.to(device) - mean) / std
        output.append(model(batch).float().cpu().numpy())
    return np.concatenate(output)


ids = cohort["Tissue_ID"].to_numpy()

if EMBEDDING_PATH.exists():
    cached = np.load(EMBEDDING_PATH, allow_pickle=True)
    assert list(cached["tissue_ids"]) == list(ids), "Cached embeddings out of date"
    embeddings = cached["embeddings"]
    print("Loaded cached embeddings.")
else:
    start = time.time()
    embeddings = extract_embeddings(cohort["Selected_Image"].tolist())
    EMBEDDING_PATH.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(EMBEDDING_PATH, tissue_ids=ids, embeddings=embeddings)
    print(f"Extracted in {(time.time() - start) / 60:.1f} min; saved to {EMBEDDING_PATH}")

print("Embedding matrix:", embeddings.shape)
assert np.isfinite(embeddings).all()

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to C:\Users\pulit/.cache\torch\hub\checkpoints\resnet50-11ad3fa6.pth


Extracted in 2.5 min; saved to C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\image\data\processed\embeddings\resnet50_imagenet_fullres.npz
Embedding matrix: (1009, 2048)


## 3. Feature Matrix and Model Arms

In [4]:
EMBEDDING_COLUMNS = [f"emb_{i}" for i in range(embeddings.shape[1])]

X = pd.concat([
    cohort[ms.ALL_FEATURES].reset_index(drop=True),
    pd.DataFrame(embeddings, columns=EMBEDDING_COLUMNS),
], axis=1)


def image_branch(n_components):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("pca", PCA(n_components=n_components, random_state=SEED)),
    ])


def make_arm(tabular, image_components):
    branches = []
    if tabular:
        branches.append(("tabular", ms.make_preprocessor(), ms.ALL_FEATURES))
    if image_components:
        branches.append(("image", image_branch(image_components), EMBEDDING_COLUMNS))
    return Pipeline([
        ("features", ColumnTransformer(branches, remainder="drop")),
        ("classifier", RandomForestClassifier(**RF_PARAMS)),
    ])


ARMS = {
    "Tabular": make_arm(tabular=True, image_components=None),
    f"Hybrid (PCA {PRIMARY_PCA})": make_arm(tabular=True, image_components=PRIMARY_PCA),
    f"Image only (PCA {PRIMARY_PCA})": make_arm(tabular=False, image_components=PRIMARY_PCA),
    **{
        f"Hybrid (PCA {k})": make_arm(tabular=True, image_components=k)
        for k in SENSITIVITY_PCA
    },
}
PRIMARY, BASELINE = f"Hybrid (PCA {PRIMARY_PCA})", "Tabular"

for name, arm in ARMS.items():
    n_features = clone(arm).named_steps["features"].fit(X.iloc[:200]).transform(X.iloc[:200]).shape[1]
    print(f"{name:20s} -> {n_features} model inputs")

Tabular              -> 62 model inputs
Hybrid (PCA 16)      -> 78 model inputs


Image only (PCA 16)  -> 16 model inputs


Hybrid (PCA 8)       -> 70 model inputs
Hybrid (PCA 32)      -> 94 model inputs


## 4. Leakage Checks

In [5]:
fold_splits = {
    repeat: list(ms.iter_donor_folds(folds, cohort["Tissue_ID"], repeat))
    for repeat in range(ms.N_REPEATS)
}

for repeat, splits in fold_splits.items():
    tested = np.concatenate([test for _, test in splits])
    assert sorted(tested) == list(range(len(cohort))), "Each tissue tested once per repeat"
    for train, test in splits:
        assert set(train).isdisjoint(test)
        train_donors = set(cohort.loc[train, "Donor_Group"])
        test_donors = set(cohort.loc[test, "Donor_Group"])
        assert train_donors.isdisjoint(test_donors), "Donor crosses fold boundary"

old_holdout = pd.read_csv(ms.SPLIT_PATH)
print("Every tissue is tested exactly once per repeat.")
print("No donor appears in both training and test data in any of the 25 folds.")
print("Transforms (imputation, scaling, encoding, PCA) are fitted per fold via clone(pipeline).fit(train).")
print(f"The old holdout split ({len(old_holdout)} rows) is not used here.")

Every tissue is tested exactly once per repeat.
No donor appears in both training and test data in any of the 25 folds.
Transforms (imputation, scaling, encoding, PCA) are fitted per fold via clone(pipeline).fit(train).
The old holdout split (1036 rows) is not used here.


## 5. Cross-Validated Evaluation

In [6]:
METRICS = {
    "Macro F1": lambda t, p: f1_score(t, p, average="macro"),
    "Balanced accuracy": balanced_accuracy_score,
    "Accuracy": accuracy_score,
}

fold_rows = []
oof = {(name, r): np.empty(len(cohort), dtype=object)
       for name in ARMS for r in range(ms.N_REPEATS)}

start = time.time()
for repeat, splits in fold_splits.items():
    for fold, (train, test) in enumerate(splits):
        for name, arm in ARMS.items():
            model = clone(arm).fit(X.iloc[train], y[train])
            pred = model.predict(X.iloc[test])
            oof[(name, repeat)][test] = pred
            fold_rows.append({
                "Arm": name, "Repeat": repeat, "Fold": fold,
                "N_train": len(train), "N_test": len(test),
                **{m: fn(y[test], pred) for m, fn in METRICS.items()},
            })

fold_scores = pd.DataFrame(fold_rows)
print(f"{len(fold_scores)} fold evaluations in {(time.time() - start) / 60:.1f} min")

125 fold evaluations in 2.5 min


In [7]:
summary = (
    fold_scores.groupby("Arm")[list(METRICS)]
    .agg(["mean", "std"])
    .reindex(ARMS)
)
print("Fold-level scores (mean and SD over 25 folds):")
print(summary.round(4).to_string())

Fold-level scores (mean and SD over 25 folds):
                    Macro F1         Balanced accuracy         Accuracy        
                        mean     std              mean     std     mean     std
Arm                                                                            
Tabular               0.8490  0.0213            0.8412  0.0233   0.8496  0.0196
Hybrid (PCA 16)       0.8587  0.0245            0.8408  0.0279   0.8611  0.0241
Image only (PCA 16)   0.4418  0.0332            0.4415  0.0332   0.5045  0.0327
Hybrid (PCA 8)        0.8446  0.0255            0.8292  0.0289   0.8464  0.0245
Hybrid (PCA 32)       0.8522  0.0233            0.8330  0.0254   0.8549  0.0234


## 6. Paired Comparison: Hybrid vs Tabular

Both arms are evaluated on the same held-out donors in every fold, so differences are paired.

Folds overlap across repeats and training sets overlap across folds, so the 25 fold differences are not independent. Two summaries account for this:

1. **Repeat-level:** out-of-fold predictions are pooled within each repeat, giving 5 repeat-level scores per arm.
2. **Corrected repeated-CV t-test** (Nadeau & Bengio, 2003; Bouckaert & Frank, 2004): inflates the variance by the test-to-train size ratio, with 24 degrees of freedom.

In [8]:
def corrected_ttest(differences, n_train, n_test):
    d = np.asarray(differences)
    n = len(d)
    se = np.sqrt((1 / n + n_test / n_train) * d.var(ddof=1))
    t = d.mean() / se
    p = 2 * stats.t.sf(abs(t), df=n - 1)
    half_width = stats.t.ppf(0.975, df=n - 1) * se
    return d.mean(), (d.mean() - half_width, d.mean() + half_width), t, p


def paired_comparison(arm, baseline=BASELINE):
    a = fold_scores[fold_scores["Arm"] == arm].set_index(["Repeat", "Fold"])
    b = fold_scores[fold_scores["Arm"] == baseline].set_index(["Repeat", "Fold"])
    rows = []
    for metric in METRICS:
        d = a[metric] - b[metric]
        mean, ci, t, p = corrected_ttest(
            d, a["N_train"].mean(), a["N_test"].mean()
        )
        rows.append({
            "Metric": metric,
            "Mean difference": mean,
            "95% CI (corrected)": f"[{ci[0]:+.4f}, {ci[1]:+.4f}]",
            "Folds improved": f"{(d > 0).sum()}/{len(d)}",
            "t": t,
            "p (corrected)": p,
        })
    return pd.DataFrame(rows)


print(f"{PRIMARY} minus {BASELINE}, per fold:")
print(paired_comparison(PRIMARY).round(4).to_string(index=False))

Hybrid (PCA 16) minus Tabular, per fold:
           Metric  Mean difference 95% CI (corrected) Folds improved       t  p (corrected)
         Macro F1           0.0098 [-0.0118, +0.0313]          16/25  0.9354         0.3589
Balanced accuracy          -0.0004 [-0.0215, +0.0207]          11/25 -0.0367         0.9710
         Accuracy           0.0115 [-0.0131, +0.0361]          15/25  0.9625         0.3454


In [9]:
repeat_rows = []
for name in ARMS:
    for repeat in range(ms.N_REPEATS):
        pred = oof[(name, repeat)]
        repeat_rows.append({
            "Arm": name, "Repeat": repeat,
            **{m: fn(y, pred) for m, fn in METRICS.items()},
        })
repeat_scores = pd.DataFrame(repeat_rows)

pivot = repeat_scores.pivot(index="Repeat", columns="Arm", values="Macro F1")[
    [BASELINE, PRIMARY]
]
pivot["Difference"] = pivot[PRIMARY] - pivot[BASELINE]
print("Repeat-level macro F1 (pooled out-of-fold predictions):")
print(pivot.round(4).to_string())
print(f"\nRepeats where the hybrid is better: {(pivot['Difference'] > 0).sum()}/{ms.N_REPEATS}")

print("\nRepeat-level means (all arms):")
print(repeat_scores.groupby("Arm")[list(METRICS)].agg(["mean", "std"])
      .reindex(ARMS).round(4).to_string())

Repeat-level macro F1 (pooled out-of-fold predictions):


Arm     Tabular  Hybrid (PCA 16)  Difference
Repeat                                      
0        0.8497           0.8557      0.0060
1        0.8537           0.8599      0.0061
2        0.8431           0.8611      0.0180
3        0.8471           0.8547      0.0076
4        0.8525           0.8642      0.0117

Repeats where the hybrid is better: 5/5

Repeat-level means (all arms):
                    Macro F1         Balanced accuracy         Accuracy        
                        mean     std              mean     std     mean     std
Arm                                                                            
Tabular               0.8492  0.0043            0.8413  0.0051   0.8496  0.0048
Hybrid (PCA 16)       0.8591  0.0039            0.8409  0.0043   0.8611  0.0045
Image only (PCA 16)   0.4419  0.0126            0.4415  0.0125   0.5045  0.0112
Hybrid (PCA 8)        0.8451  0.0084            0.8293  0.0075   0.8464  0.0077
Hybrid (PCA 32)       0.8527  0.0042            0.8

## 7. Per-Grade Behaviour

In [10]:
def pooled_report(name):
    true = np.tile(y, ms.N_REPEATS)
    pred = np.concatenate([oof[(name, r)] for r in range(ms.N_REPEATS)])
    report = classification_report(
        true, pred, labels=ms.CLASSES, output_dict=True, zero_division=0
    )
    return pd.DataFrame(report).T.loc[ms.CLASSES, ["precision", "recall", "f1-score"]]

print("Per-grade scores, pooled over all repeats:")
print(pd.concat(
    {name: pooled_report(name) for name in [BASELINE, PRIMARY, f"Image only (PCA {PRIMARY_PCA})"]},
    axis=1,
).round(3).to_string())

for name in [BASELINE, PRIMARY]:
    cm = confusion_matrix(y, oof[(name, 0)], labels=ms.CLASSES)
    print(f"\n{name} — confusion matrix, repeat 0 (rows = actual):")
    print(pd.DataFrame(cm, index=[f"Actual {c}" for c in ms.CLASSES],
                       columns=[f"Pred {c}" for c in ms.CLASSES]))

changed = pd.Series(
    [(oof[(BASELINE, r)] != oof[(PRIMARY, r)]).sum() for r in range(ms.N_REPEATS)]
)
print(f"\nPredictions changed by adding images: {changed.mean():.0f} of {len(cohort)} per repeat on average")

Per-grade scores, pooled over all repeats:
     Tabular                 Hybrid (PCA 16)                 Image only (PCA 16)                
   precision recall f1-score       precision recall f1-score           precision recall f1-score
A+     0.878  0.886    0.882           0.866  0.920    0.892               0.629  0.655    0.642
A      0.763  0.787    0.775           0.797  0.783    0.790               0.365  0.345    0.354
B      0.935  0.851    0.891           0.987  0.819    0.895               0.335  0.325    0.330

Tabular — confusion matrix, repeat 0 (rows = actual):
           Pred A+  Pred A  Pred B
Actual A+      469      59       2
Actual A        59     248       7
Actual B         8      17     140

Hybrid (PCA 16) — confusion matrix, repeat 0 (rows = actual):
           Pred A+  Pred A  Pred B
Actual A+      484      46       0
Actual A        68     246       0
Actual B         9      22     134

Predictions changed by adding images: 109 of 1009 per repeat on average


## 8. Sensitivity to the Number of PCA Components

These arms were specified in advance and are reported for robustness only; the primary result above uses 16 components.

In [11]:
sensitivity = []
for k in [SENSITIVITY_PCA[0], PRIMARY_PCA, SENSITIVITY_PCA[1]]:
    table = paired_comparison(f"Hybrid (PCA {k})").set_index("Metric")
    sensitivity.append({
        "PCA components": k,
        "Macro F1 difference": table.loc["Macro F1", "Mean difference"],
        "95% CI (corrected)": table.loc["Macro F1", "95% CI (corrected)"],
        "Folds improved": table.loc["Macro F1", "Folds improved"],
        "p (corrected)": table.loc["Macro F1", "p (corrected)"],
    })
print("Hybrid minus Tabular, macro F1:")
print(pd.DataFrame(sensitivity).round(4).to_string(index=False))

Hybrid minus Tabular, macro F1:
 PCA components  Macro F1 difference 95% CI (corrected) Folds improved  p (corrected)
              8              -0.0044 [-0.0260, +0.0172]          11/25         0.6776
             16               0.0098 [-0.0118, +0.0313]          16/25         0.3589
             32               0.0032 [-0.0209, +0.0274]          16/25         0.7837


## 9. Save Out-of-Fold Predictions

Saved for the secondary (missing-information) analysis and later error analysis.

In [12]:
oof_df = pd.DataFrame({
    "Tissue_ID": np.tile(cohort["Tissue_ID"], ms.N_REPEATS),
    "Donor_Group": np.tile(cohort["Donor_Group"], ms.N_REPEATS),
    "Repeat": np.repeat(range(ms.N_REPEATS), len(cohort)),
    "Actual": np.tile(y, ms.N_REPEATS),
})
for name in ARMS:
    oof_df[name] = np.concatenate([oof[(name, r)] for r in range(ms.N_REPEATS)])

oof_df.to_csv(OOF_PATH, index=False)
fold_scores.to_csv(ms.PROCESSED_DIR / "hybrid_fold_scores.csv", index=False)
print("Saved:", OOF_PATH)
print("Saved:", ms.PROCESSED_DIR / "hybrid_fold_scores.csv")

Saved: C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed\hybrid_oof_predictions.csv
Saved: C:\Users\pulit\OneDrive\Desktop\University\Y4S1\research\xai-cornea-grading\ml\grading\data\processed\hybrid_fold_scores.csv


## Summary

### Primary result (all 24 features available)

| Arm | Macro F1 | Balanced accuracy | Accuracy |
|---|---:|---:|---:|
| Tabular | 0.849 ± 0.021 | 0.841 ± 0.023 | 0.850 ± 0.020 |
| Hybrid (PCA 16) | 0.859 ± 0.025 | 0.841 ± 0.028 | 0.861 ± 0.024 |
| Image only (PCA 16) | 0.442 ± 0.033 | 0.442 ± 0.033 | 0.505 ± 0.033 |

Fold-level mean ± SD over 5 × 5 donor-grouped folds.

**Paired difference, hybrid minus tabular:**

- Macro F1 +0.010, corrected 95% CI [−0.012, +0.031], p = 0.36; better in 16 of 25 folds.
- Higher in all 5 repeats (+0.006 to +0.018). The repeats reuse the same tissues, so this shows the result is stable across fold assignments, not that it is significant.
- Balanced accuracy unchanged (−0.000).

**Conclusion:** With all clinical and morphology features available, adding frozen ImageNet ResNet-50 embeddings produced a small, non-significant macro F1 gain. This experiment does not show that images add information beyond the tabular features.

### Supporting observations

1. **The embeddings carry some grade information.** Image-only macro F1 is 0.44, above the ~0.33 chance level but far below tabular. Most of this signal is probably already captured by the Konan measurements derived from the same images.
2. **The gain is not robust to the number of components.** PCA 8 gives −0.004 and PCA 32 gives +0.003; 16 components is the best of the three, which is consistent with noise around zero.
3. **Images shift errors rather than remove them.** About 109 of 1,009 predictions change per repeat, but the net gain is small. A+ recall rises (0.886 → 0.920) and B precision rises (0.935 → 0.987), while **B recall falls (0.851 → 0.819)**. More B tissues are called A, which is the clinically less safe direction, because poorer tissue is rated higher. Balanced accuracy is therefore unchanged.

### Limitations

- The extractor is ImageNet-pretrained and frozen. A representation adapted to specular microscopy (fine-tuning, or self-supervised pretraining on the 1,024 images) may capture more.
- The RF hyperparameters were tuned in notebook 05 for the tabular features with non-grouped CV on tissues that now also appear in the test folds. This affects both arms but may slightly favour the tabular arm.
- Donor groups are inferred, not confirmed against a donor register.

### Next step

The secondary scenario: test whether image embeddings help when slit-lamp findings or Konan measurements are unavailable, using the saved out-of-fold predictions (natural complete vs incomplete records) and feature-masking experiments on the same folds.

## Addendum (after notebook 08)

The pooled results above combine two subsets with different recorded grading patterns. On 2023 tissues the tabular model reaches macro F1 0.992; on tissues from other years it reaches 0.729. The pooled comparison is valid as stated, but it is dominated by 2023 tissues where there is little room for improvement. By year, the hybrid minus tabular difference is −0.022 (2023) and +0.036 (other years, p = 0.078, unadjusted). The latter was found after seeing the data and is hypothesis-generating only. See `08_missing_information.ipynb`, Section 10.